# Single annotated autoencoder analysis
Each section runs one explicit analysis over one prepared model cache.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

current_path = Path.cwd().resolve()
repository_root = next(path for path in (current_path, *current_path.parents) if (path / 'pyproject.toml').is_file())
os.chdir(repository_root)

In [ ]:
PROJECT_PATH = 'workspace'
DEFAULT_IMAGE = 'kidney-pilot-v2'
MODEL_NAME = 'kidney-annotation-autoencoder'
HEAD_NAME = 'molecule_primary'
BATCH_SIZE = 8
RANDOM_SEED = 1912

In [ ]:
from msi_autoencoder_wrapper import MSIAutoEncoderWrapper
from msi_autoencoder_wrapper.analysis import AutoencoderAnalysis
from msi_autoencoder_wrapper.visualization import VisualizationTheme

wrapper = MSIAutoEncoderWrapper(project_path=PROJECT_PATH)
wrapper.workspace.set_default_image_path(DEFAULT_IMAGE)
theme = VisualizationTheme(model_overrides={MODEL_NAME: '#2563EB'})
analysis = AutoencoderAnalysis(wrapper, model_name=MODEL_NAME, theme=theme)
prepared = analysis.prepare(batch_size=BATCH_SIZE)
mass_axis = wrapper.active_context.binner.GetXAxis()

## Reconstruction metrics and spatial error

In [ ]:
metric_table = pd.DataFrame(analysis.reconstruction.compare_metrics(
    metrics=['mse', 'mae', 'masserstein', 'cosine_similarity', 'spectral_angle', 'tic_error'],
))
metric_table

In [ ]:
analysis.reconstruction.plot_distribution('mse')
analysis.reconstruction.plot_distribution('spectral_angle')

In [ ]:
analysis.reconstruction.plot_error_images('mse', annotation_filter='all', target_field='molecule')
analysis.reconstruction.plot_error_images('mse', annotation_filter='annotated', target_field='molecule')
analysis.reconstruction.plot_error_images('mse', annotation_filter='unannotated', target_field='molecule')

In [ ]:
analysis.reconstruction.plot_selected_spectra_comparison(metric='mse', selection='best', count=5)
analysis.reconstruction.plot_selected_spectra_comparison(metric='mse', selection='median', count=5)
analysis.reconstruction.plot_selected_spectra_comparison(metric='mse', selection='worst', count=5)

In [ ]:
worst_features = pd.DataFrame(analysis.reconstruction.rank_features(metric='mse', top_n=10))
analysis.reconstruction.plot_feature_error_overview(metric='mse', quantiles=(0.05, 0.95), top_n=10)
worst_features

In [ ]:
worst_feature = int(np.argmax(prepared.feature_metrics['feature_mse']))
selected_mz = float(mass_axis[worst_feature])
analysis.reconstruction.plot_ion_images(selected_mz, tolerance=0.02, target_field='molecule')

In [ ]:
viewer = analysis.reconstruction.ion_viewer(
    tolerance=0.02, target_field='molecule', mz_values=worst_features['mz'].to_numpy(),
)
viewer.widget(initial_index=0)

## Latent space and molecule classes

In [ ]:
analysis.latent.plot_target_projection('molecule', method='pca', mode='overlay', head_name=HEAD_NAME, random_seed=RANDOM_SEED)
analysis.latent.plot_target_projection('molecule', method='pca', mode='panels', head_name=HEAD_NAME, random_seed=RANDOM_SEED)

In [ ]:
analysis.latent.plot_components(range(prepared.arrays['latents'].shape[1]), source='latent')
analysis.latent.plot_components(range(min(8, prepared.arrays['latents'].shape[1])), source='pca')

## Molecule-head probabilities, correctness, and class metrics

In [ ]:
class_metrics = pd.DataFrame(analysis.heads.class_metrics(HEAD_NAME, threshold=0.5))
display_classes = class_metrics.sort_values('f1').head(6)['class_index'].astype(int).tolist()
analysis.heads.plot_class_overview(HEAD_NAME, display_classes, threshold=0.5)
class_metrics